# Day 068 · 价值因子
**Value Factors** · 阶段 P3 · 数据与因子工程

> 这一节我们正式认识因子家族里最古老、最朴素的一员:价值因子。一句话,价值因子干的事就是「捡便宜」,在一堆股票里挑那些价格相对它真实身价更低的,赌它们迟早被市场重新看见。怎么衡量便宜?最常用三把尺子:市盈率,也就是市值除以净利润,等于说花多少年的利润才能把买它的钱赚回来;市净率,也就是市值除以净资产,等于说花几块钱去买它账上1 块钱的家底;还有 EV 除以息税折旧前利润,它能把负债不一样的公司拉到同一条起跑线上比。这一节我们用12只跨行业的真实股票,专门拿市净率这把尺子,把最便宜的一组和最贵的一组分开,各自月度调仓回测10 年,亲眼看便宜到底香不香。但便宜从来不是免费的午餐:有的票一直便宜、却一直不涨甚至一路阴跌,这就是价值因子最大的暗坑,也就是价值陷阱。便宜,往往是有原因的。

---

**课件生成日期:** 2026-06-26  ·  **建议学习时长:** 22 分钟

学习路径建议:1)先看视频建立直觉 → 2)阅读本 notebook 跑代码 → 3)看 PDF 课件复习要点 → 4)做自测题

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有需要的 Python 包,缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续下面的代码

> 这一格只在第一次跑要等几十秒,后面再开 notebook 就秒过。

In [1]:
# === 环境自检 + 自动安装(运行此单元格即可) ===
# 检测缺失的库 → 自动 pip 安装 → 注入中文字体 → 一行命令搞定
import importlib
import subprocess
import sys
import os

REQUIRED = ["baostock", "matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels", "yfinance"]
PIP_NAME = {
    "sklearn": "scikit-learn",
    "cv2": "opencv-python",
    "PIL": "Pillow",
    "bs4": "beautifulsoup4",
    "yaml": "PyYAML",
}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))

if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,正在自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置(让 matplotlib 不出乱码) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

CJK_FONT_PATHS = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",  # Linux/WSL
    "C:/Windows/Fonts/msyh.ttc",                               # Windows 微软雅黑
    "C:/Windows/Fonts/simhei.ttf",                             # Windows 黑体
    "/System/Library/Fonts/PingFang.ttc",                      # macOS 苹方
    "/System/Library/Fonts/STHeiti Medium.ttc",                # macOS 黑体
]
for p in CJK_FONT_PATHS:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Microsoft YaHei",
                                    "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪 — 现在可以跑下面的代码单元格")


✓ 所有 9 个必需库已就绪
✓ 中文字体已加载:NotoSansCJK-Regular.ttc
✓ 环境就绪 — 现在可以跑下面的代码单元格


## 学习目标

- 搞懂价值因子在干一件什么事:在一堆股票里挑相对真实身价更便宜的,赌它被低估、迟早会被重新定价
- 用大白话+小例子说清三把估值尺子:市盈率(市值÷净利润)、市净率(市值÷净资产)、EV÷息税折旧前利润(跨负债可比)
- 亲手把12只票按市净率分成低PB(便宜)组和高PB(贵)组,各自月度调仓回测10 年,看便宜组长期到底跑赢还是跑输
- 理解价值因子的脾气:便宜不等于立刻涨,它有时灵有时不灵,是个需要熬、需要耐心的慢因子
- 认清价值陷阱:有的票常年便宜却一路下跌,便宜是有原因的,光看一个低PB就买进去会被套住

## 历史背景:老周3 年死守一只「全市场最便宜」的低PB股,越跌越买,结果套了3 年才明白便宜也是有原因的

老周是个有10 年股龄的老散户,信奉一句话:买股票就是买便宜货。前几年他翻遍全市场,挑中一只市净率只有零点几倍的票,也就是说,它账上1 块钱的家底,市场只肯出几毛钱来买,在他眼里简直是地板价捡黄金。他重仓买入,心想这么便宜还能跌到哪去。结果第一年跌了,他不慌,反而觉得更便宜了,加仓;第2 年又跌,他还是加仓,嘴里念叨着价值终会回归;到第3 年,这票的市净率更低了,可股价也更低了,他账户已经亏掉一大块。后来一个做量化的朋友帮他复了个盘,翻出这只票10 年的曲线给他看:股价一路向下,市净率也一路趴在地板上从没起来过。朋友说,你看,它不是某一天突然便宜,它是一直便宜、一直跌,这叫价值陷阱。便宜往往是有原因的,可能行业在萎缩,可能它的家底在持续缩水,市场不是看不见它便宜,是市场觉得它该这么便宜。老周这才明白,光盯着一个低市净率买股票,捡的不一定是黄金,也可能是一直往下掉的飞刀。这一节,我们就用12只真实股票,把老周踩的这个价值陷阱,完整还原一遍。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. 1. 价值因子:说白了就是科学地捡便宜

价值因子是因子世界里最古老、最好懂的一个。它的逻辑特别朴素:在一堆股票里,找出那些价格相对它真实身价偏低的,也就是被市场暂时看扁、卖便宜了的,然后赌它迟早会被重新看见、价格回归。打个比方,同样一台九成新的2 手车,有人标价8 万有人标价5 万,价值因子干的就是去找那台5 万的,赌它其实也值8 万。它不预测明天涨跌,它只挑当下相对便宜的一篮子,长期持有等价值回归。问题在于:怎么量化一只票到底便宜不便宜?这就要用到下面几把估值尺子。


### 2. 2. 市盈率 PE:几年利润能回本

第一把尺子叫市盈率,英文 PE,算法是公司市值除以一年的净利润。它回答一个特别直白的问题:照现在的赚钱速度,买下整家公司要几年才能靠利润回本。举个最小的例子:一家公司市值一百亿,一年净赚10 亿,那市盈率就是100除以十,等于10 倍,等于说10 年的利润就能把买它的钱赚回来。如果另一家市值同样一百亿、一年只赚两亿,市盈率就是50 倍,要50 年才回本,显然贵得多。所以市盈率越低,通常意味着越便宜。注意一个坑:亏损的公司净利润是负的,算出来的市盈率没有意义,要单独剔除。


### 3. 3. 市净率 PB:花几块钱买它1 块钱家底

第二把尺子叫市净率,英文 PB,算法是公司市值除以净资产。净资产你可以理解成这家公司的家底,就是把它所有的厂房、设备、现金、存货加起来,再减掉欠别人的钱,剩下真正属于股东的那部分。市净率回答的是:你花几块钱,去买它账上1 块钱的家底。比如市净率是3 倍,等于说它1 块钱的净资产,你要花3 块钱买;如果市净率只有零点8 倍,等于说它1 块钱的家底,你花八毛就能买到,听起来像打折。这一节我们专门用市净率来选股。一个常识:银行、地产、钢铁、煤炭这类重资产行业,市净率天然就低;消费、医药、科技这类轻资产、高成长行业,市净率天然就高。


### 4. 4. EV÷EBITDA:把负债不同的公司拉到同一起跑线

第三把尺子名字唬人,叫 EV 除以息税折旧前利润,但思路不难。前面的市盈率有个小毛病:它只看股东这部分,没把公司借的债算进去。两家公司利润一样,但一家没欠债、一家欠了一屁股债,光看市盈率会觉得它们一样贵,其实欠债多的那家风险大得多。EV,也就是企业价值,等于市值加上净负债,把欠的债也算进收购成本里。再除以息税折旧前利润,等于说把利息、税、折旧这些会计差异都拨开,只看公司本身的赚钱能力。这把尺子的好处是:负债结构差很多的公司,用它来比才公平,所以做并购、比重资产公司时特别常用。


### 5. 5. 价值陷阱:便宜,往往是有原因的

这是价值因子最该警惕的暗坑。我们总觉得便宜就是机会,但有一类票它一直便宜、却一直不涨甚至一路阴跌,你越买越亏,这就叫价值陷阱。打个比方,一件衣服标价从五折降到三折再降到一折,不一定是越来越划算,也可能是它过季了、有破洞、根本没人要,商家才一降再降。股票也一样:一只票市净率长期趴在地板上,可能不是市场犯傻没发现它便宜,而是市场早看明白了,它所在的行业在萎缩、它的家底在缩水、它的赚钱能力在退化,所以市场就觉得它该这么便宜。光看一个低市净率就冲进去,捡的可能不是黄金,而是还在往下掉的飞刀。怎么避开?要结合它便不便宜的原因,看家底是真值钱还是在缩水,而不是只盯一个静态的低估值数字。


## 实操:价值因子:用市净率把12只票分成低PB(便宜)组和高PB(贵)组,月度调仓回测10 年,看便宜组长期跑赢还是跑输,并揪出常年便宜却一路下跌的价值陷阱实例

下面这段代码跟视频里讲解的 highlights 是一致的,可以**直接 Run All** 看结果。

**依赖安装:**
```bash
pip install pandas numpy matplotlib yfinance akshare statsmodels
```


In [2]:
# day_068_value_factors.py — 价值因子:低 PB 分组回测 + 价值陷阱
# 真名上屏:baostock / query_history_k_data_plus → pbMRQ(市净率,价值因子) / nsmallest 低PB(便宜)组 / nlargest 高PB(贵)组 / 月末调仓
# 核心类比:PB=市值÷净资产=「花几块钱买它一块钱的家底」;低PB=便宜,但便宜≠立刻涨,有的票一直便宜一直跌就是价值陷阱(便宜是有原因的)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import baostock as bs


def _data_path(_name):
    # 铁律62:data/ 放在 notebook 文件夹里。仓库根(run_lab)存取 out/notebook/data/;
    # 原版 notebook 在 out/notebook/ 用 data/;中国版在 out/notebook/cn/ 用 ../data/
    from pathlib import Path as _P
    _here = _P.cwd()
    for _b in [_here / 'data', _here / '..' / 'data', _here / 'out' / 'notebook' / 'data', _here / '..' / '..' / 'data', _here / '..' / '..' / '..' / 'data']:
        if (_b / _name).exists():
            return str(_b / _name)
    if (_here / 'out' / 'notebook').exists():
        _t = _here / 'out' / 'notebook' / 'data'
    elif _here.name == 'cn':
        _t = _here / '..' / 'data'
    else:
        _t = _here / 'data'
    _t.mkdir(parents=True, exist_ok=True)
    return str(_t / _name)


pd.set_option('display.width', 160)
plt.rcParams['axes.unicode_minus'] = False

# ==== 标的池:12 只跨行业票,PB 从低到高铺开 ====
# 银行/地产/钢铁/煤炭天然低PB(便宜),消费/医药/科技天然高PB(贵)
STOCKS = {
    '建设银行': 'sh.601939', '中国银行': 'sh.601988', '华夏银行': 'sh.600015',
    '宝钢股份': 'sh.600019', '陕西煤业': 'sh.601225', '金地集团': 'sh.600383',
    '上汽集团': 'sh.600104', '爱尔眼科': 'sz.300015', '迈瑞医疗': 'sz.300760',
    '汇川技术': 'sz.300124', '恒立液压': 'sh.601100', '通策医疗': 'sh.600763',
}
START, END = '2015-01-01', '2024-12-31'
CSV_PX = _data_path('D068_value.csv')   # 前复权收盘(算策略真实收益)
CSV_PB = _data_path('D068_pb.csv')      # 市净率 pbMRQ(价值因子,日频自带,当天即已知)


# ==== 0. 拉数据:前复权收盘 + 市净率 pbMRQ。部分票上市晚 -> 用各自可得历史 ====
def _fetch():
    bs.login()
    px, pb = {}, {}
    for name, code in STOCKS.items():
        rs = bs.query_history_k_data_plus(code, 'date,close,pbMRQ', start_date=START, end_date=END, frequency='d', adjustflag='2')
        r = []
        while rs.error_code == '0' and rs.next():
            r.append(rs.get_row_data())
        s = pd.DataFrame(r, columns=['date', 'close', 'pbMRQ'])
        s['close'] = pd.to_numeric(s['close'], errors='coerce')
        s['pbMRQ'] = pd.to_numeric(s['pbMRQ'], errors='coerce')
        s = s.set_index('date')
        px[name] = s['close']
        pb[name] = s['pbMRQ']
    bs.logout()
    return pd.DataFrame(px), pd.DataFrame(pb)


if os.path.exists(CSV_PX) and os.path.exists(CSV_PB):
    print('[数据] 从本地 CSV 读取 收盘 + 市净率')
    px = pd.read_csv(CSV_PX, parse_dates=['date']).set_index('date')
    pb = pd.read_csv(CSV_PB, parse_dates=['date']).set_index('date')
else:
    print('[数据] baostock 拉取 12 只票 前复权收盘 + 市净率 pbMRQ ...')
    px, pb = _fetch()
    px.index.name = 'date'
    pb.index.name = 'date'
    px.to_csv(CSV_PX)             # 拉完保存 CSV(铁律62)
    pb.to_csv(CSV_PB)
    print('[数据] 已存 CSV ->', CSV_PX)

px.index = pd.to_datetime(px.index)   # 两条路径(CSV/fetch)统一成日期索引
pb.index = pd.to_datetime(pb.index)
pb = pb.reindex(px.index)

print('\n==== 价值因子:低 PB(便宜)分组回测 + 价值陷阱 ====')
print('标的池 %d 只 · %s ~ %s' % (len(STOCKS), px.index[0].date(), px.index[-1].date()))


def maxdd(equity):
    # 最大回撤(%):从历史最高点跌下来最深的那一下
    e = pd.Series(equity).dropna()
    if len(e) < 2:
        return 0.0
    peak = e.cummax()
    return float((e / peak - 1.0).min() * 100)


def ann_return(equity, n_months):
    # 年化收益(%):把总收益按月数折算成每年
    e = pd.Series(equity).dropna()
    if len(e) < 2 or n_months <= 0:
        return 0.0
    yrs = n_months / 12.0
    return float((e.iloc[-1] ** (1.0 / yrs) - 1.0) * 100)


# ==== 1. 月末调仓:低PB(便宜)组 / 高PB(贵)组 / 基准等权 三条净值 ====
monthly = px.resample('ME').last()                 # 月末收盘
mret = monthly.pct_change().shift(-1)              # 本月末建仓 -> 下月末的收益
pb_m = pb.resample('ME').last()                    # 月末市净率(日频自带,当天即已知,无需滞后)


def backtest(mode):
    # mode: 'low'=低PB便宜组 / 'high'=高PB贵组 / 'base'=等权全部
    eq, dates = [1.0], [monthly.index[0]]
    yearly = {}   # 自然年 -> 这一年的累计因子(年末/年初)
    cur_year = monthly.index[0].year
    year_acc = 1.0
    for i in range(len(monthly.index) - 1):
        t = monthly.index[i]
        v = pb_m.loc[t]
        v = v[(v.notna()) & (v > 0)]                # 剔除缺失/非正的 PB
        r_next = mret.loc[t]
        n = len(v)
        if n >= 2:
            k = max(3, n // 3)
            k = min(k, n)
            if mode == 'low':
                sel = v.nsmallest(k).index           # 便宜:PB 最小的一组
            elif mode == 'high':
                sel = v.nlargest(k).index            # 贵:PB 最大的一组
            else:
                sel = v.index                        # 基准:等权全部
            r = r_next[sel].mean()
        else:
            r = r_next.mean()
        r = 0.0 if pd.isna(r) else r
        eq.append(eq[-1] * (1 + r))
        nxt = monthly.index[i + 1]
        dates.append(nxt)
        # 按自然年记账(用建仓月所属年)
        yr = t.year
        if yr != cur_year:
            yearly[cur_year] = year_acc - 1.0
            cur_year = yr
            year_acc = 1.0
        year_acc *= (1 + r)
    yearly[cur_year] = year_acc - 1.0
    return pd.Series(eq, index=dates), yearly


eq_low, yr_low = backtest('low')
eq_high, yr_high = backtest('high')
eq_base, yr_base = backtest('base')
n_months = len(monthly.index) - 1

print('\n[低PB(便宜)组 / 高PB(贵)组 / 基准等权 — 同一段 %d 个月]' % n_months)
for tag, eq in [('低PB便宜组', eq_low), ('高PB贵组', eq_high), ('基准等权', eq_base)]:
    tot = (eq.iloc[-1] - 1) * 100
    print('%s  总收益 %+.1f%%   年化 %+.1f%%   最大回撤 %.1f%%' % (
        tag, tot, ann_return(eq.values, n_months), maxdd(eq.values)))

# 价值因子需要耐心:数一数有多少个自然年 低PB组 跑输 高PB组
years = sorted(set(yr_low) & set(yr_high))
lose_years = [y for y in years if yr_low[y] < yr_high[y]]
print('\n[价值因子要耐心] 共 %d 个完整自然年,低PB(便宜)组有 %d 年跑输高PB(贵)组' % (len(years), len(lose_years)))
print('  跑输的年份:', lose_years, '-> 不是年年灵,得熬得住')

# ==== 2. 价值陷阱:常年低PB却收益差/为负的票 ====
# 对每只票:全期总收益 + 它落在「低PB便宜组」的月份占比
low_member_share = {}
for name in STOCKS:
    in_low, total = 0, 0
    for i in range(len(monthly.index) - 1):
        t = monthly.index[i]
        v = pb_m.loc[t]
        v = v[(v.notna()) & (v > 0)]
        if len(v) < 2 or name not in v.index:
            continue
        total += 1
        k = max(3, len(v) // 3)
        k = min(k, len(v))
        if name in set(v.nsmallest(k).index):
            in_low += 1
    low_member_share[name] = (in_low / total) if total else 0.0

stock_ret = {}
for name in STOCKS:
    s = px[name].dropna()
    stock_ret[name] = (s.iloc[-1] / s.iloc[0] - 1.0) * 100 if len(s) >= 2 else np.nan

pb_avg = {name: float(pb[name].dropna().mean()) for name in STOCKS}
trap_tbl = pd.DataFrame({
    '低PB组占比%': pd.Series(low_member_share) * 100,
    '常年PB': pd.Series(pb_avg),
    '全期总收益%': pd.Series(stock_ret),
}).round(2)
trap_tbl = trap_tbl.sort_values('低PB组占比%', ascending=False)
print('\n[价值陷阱筛查:常年待在低PB组(便宜常客)但收益差的票]')
print(trap_tbl.to_string())

# 价值陷阱实例:在「看着便宜」的一半票里(常年PB低于全体中位数),挑总收益最差的那只
# (不卡 低PB组占比≥50%——那样只会选出最便宜的银行,而银行2024有高股息行情翻正,不够干净;
#  地产这类"看着便宜却一路阴跌没回头"的才是教科书级价值陷阱)
med_pb = float(pd.Series(pb_avg).median())
value_side = trap_tbl[trap_tbl['常年PB'] <= med_pb]
if len(value_side) == 0:
    value_side = trap_tbl.head(6)
trap_name = value_side['全期总收益%'].idxmin()
_trap_pb = trap_tbl.loc[trap_name, '常年PB']
_trap_ret = trap_tbl.loc[trap_name, '全期总收益%']
print('\n[价值陷阱实例] %s:常年PB 仅 %.2f 倍(在12只里属便宜的一半),可全期总收益 %+.1f%%' % (
    trap_name, _trap_pb, _trap_ret))
print('  -> 看着便宜却一路阴跌、十年都没回本,便宜是有原因的,这就是价值陷阱')

# ==== 3. 三张图 ====
# 图1:三条净值曲线 低PB / 高PB / 基准
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(eq_low.index, eq_low.values, lw=2, c='#2e7d32', label='低PB便宜组')
ax.plot(eq_high.index, eq_high.values, lw=2, c='#c62828', label='高PB贵组')
ax.plot(eq_base.index, eq_base.values, lw=1.6, c='#777', ls='--', label='基准等权')
ax.axhline(1.0, c='k', lw=.8)
for eq, c in [(eq_low, '#2e7d32'), (eq_high, '#c62828'), (eq_base, '#777')]:
    ax.text(eq.index[-1], eq.iloc[-1], ' %.2f' % eq.iloc[-1], color=c, fontweight='bold', va='center')
_lo = (eq_low.iloc[-1] - 1) * 100
_hi = (eq_high.iloc[-1] - 1) * 100
_winner = '低PB(便宜)组' if _lo > _hi else '高PB(贵)组'
ax.set_title('低PB(便宜)组 vs 高PB(贵)组 vs 基准:十年下来 %s 跑赢(终值已标注)' % _winner)
ax.set_ylabel('净值(起点=1)')
ax.legend(loc='upper left')
ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig('chart_01.png', dpi=110)
plt.close()

# 图2:价值陷阱单例双轴 —— 左轴股价(一路走低) 右轴市净率(一直很低)
fig, ax = plt.subplots(figsize=(11, 6))
ps = px[trap_name].dropna()
ax.plot(ps.index, ps.values, lw=1.8, c='#333', label='股价(左轴)')
ax.set_ylabel('前复权收盘价(元)', color='#333')
ax.tick_params(axis='y', labelcolor='#333')
ax2 = ax.twinx()
pbs = pb[trap_name].dropna()
ax2.plot(pbs.index, pbs.values, lw=1.6, c='#1565c0', alpha=.8, label='市净率PB(右轴)')
ax2.axhline(1.0, c='#1565c0', ls=':', lw=1, alpha=.6)
ax2.set_ylabel('市净率 PB(倍)', color='#1565c0')
ax2.tick_params(axis='y', labelcolor='#1565c0')
ax.set_title('%s:看着便宜(PB约%.1f倍),十年总收益%+.0f%%、股价一路阴跌没回本 —— 便宜是有原因的=价值陷阱' % (
    trap_name, _trap_pb, _trap_ret))
ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig('chart_02.png', dpi=110)
plt.close()

# 图3:逐年 低PB减高PB 的超额收益柱状(正负双色)—— 价值因子有时灵有时不灵
fig, ax = plt.subplots(figsize=(11, 6))
ys = sorted(set(yr_low) & set(yr_high))
diff = [(yr_low[y] - yr_high[y]) * 100 for y in ys]
colors = ['#2e7d32' if d >= 0 else '#c62828' for d in diff]
ax.bar([str(y) for y in ys], diff, color=colors)
ax.axhline(0, c='k', lw=.8)
for i, d in enumerate(diff):
    ax.text(i, d + (1 if d >= 0 else -2), '%+.0f' % d, ha='center', fontsize=8, fontweight='bold')
_win_yrs = sum(1 for d in diff if d >= 0)
ax.set_title('逐年「低PB便宜组 − 高PB贵组」超额收益:%d 年里只有 %d 年为正 —— 价值因子有时灵有时不灵,需要耐心' % (
    len(ys), _win_yrs))
ax.set_ylabel('低PB组 − 高PB组(个百分点)')
ax.grid(alpha=.3, axis='y')
fig.tight_layout()
fig.savefig('chart_03.png', dpi=110)
plt.close()

print('\n[done] 价值因子低PB分组回测 + 价值陷阱实证完成 —— 3 图已出')

[数据] 从本地 CSV 读取 收盘 + 市净率

==== 价值因子:低 PB(便宜)分组回测 + 价值陷阱 ====
标的池 12 只 · 2015-01-05 ~ 2024-12-31

[低PB(便宜)组 / 高PB(贵)组 / 基准等权 — 同一段 119 个月]
低PB便宜组  总收益 +136.8%   年化 +9.1%   最大回撤 -31.6%
高PB贵组  总收益 +372.7%   年化 +17.0%   最大回撤 -67.1%
基准等权  总收益 +309.0%   年化 +15.3%   最大回撤 -30.6%

[价值因子要耐心] 共 10 个完整自然年,低PB(便宜)组有 6 年跑输高PB(贵)组
  跑输的年份: [2015, 2017, 2018, 2019, 2020, 2022] -> 不是年年灵,得熬得住

[价值陷阱筛查:常年待在低PB组(便宜常客)但收益差的票]
      低PB组占比%   常年PB  全期总收益%
中国银行    97.48   0.70  111.22
华夏银行    86.55   0.63   27.68
建设银行    82.35   0.82  110.71
宝钢股份    53.78   0.85   53.44
金地集团    22.69   1.12  -44.51
上汽集团    19.33   1.12   38.11
陕西煤业     0.00   1.84  403.95
爱尔眼科     0.00  15.57  529.96
迈瑞医疗     0.00  14.41  295.64
汇川技术     0.00   9.00  534.83
恒立液压     0.00   6.81  756.85
通策医疗     0.00  17.27  166.47

[价值陷阱实例] 金地集团:常年PB 仅 1.12 倍(在12只里属便宜的一半),可全期总收益 -44.5%
  -> 看着便宜却一路阴跌、十年都没回本,便宜是有原因的,这就是价值陷阱

[done] 价值因子低PB分组回测 + 价值陷阱实证完成 —— 3 图已出


## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| 反直觉·便宜组跑输贵组 | N/A | 2015到2024这10 年,低市净率便宜组总收益+136.8%(年化9.1%),反而大幅跑输高市净率贵组+372.7%(年化17%),也输给等权基准+309%。便宜的银行钢铁煤炭地产,整体输给贵的医药科技制造。A股这轮是成长牛市,价值因子并不是稳赚。 |
| 价值陷阱·金地集团 | sh.600383 | 金地集团常年市净率只有约1.1倍,在12只里属于便宜的一半,看着很便宜;可10 年总收益是-44.5%,股价从高位一路阴跌、10 年都没回本,市净率从约2倍跌到0.3倍。地产基本面恶化,便宜是有原因的——这就是典型的价值陷阱。 |
| 需要耐心·2020年最惨 | N/A | 逐年看『便宜组减贵组』的超额,十个完整自然年里只有4年为正(2016、2021、2023、2024)。最惨的2020年正值成长股狂热顶峰,便宜组单年跑输高达188个点;但从2021年起多数年份翻身。价值因子不是年年灵,得熬得住一段长时间的难受。 |
| 便宜组防守·回撤更小 | N/A | 虽然便宜组这10 年涨得少,但它最大回撤只有-31.6%,而高市净率贵组最大回撤高达-67.1%、几乎腰斩还多。便宜票跌得少也涨得少,更像防守不像进攻;价值因子在扛跌、控回撤上仍有它的价值。 |


## 常见坑

### ⚠ 01. 只盯一个低市净率就买,踩进价值陷阱

便宜往往是有原因的:行业萎缩、家底缩水、赚钱能力退化。一只票常年低PB却一路下跌,光看静态低估值冲进去就是接飞刀。

### ⚠ 02. 拿亏损股算市盈率,得到没意义的数字

净利润为负时市盈率算出来是负数或天文数字,毫无意义,做价值筛选前必须把亏损股(PE为负)单独剔除。

### ⚠ 03. 不分行业横向比估值,把银行和科技放一起比

银行/地产/钢铁/煤炭天然低PB,消费/医药/科技天然高PB;跨行业直接比谁便宜,等于拿苹果比橘子,容易把整个低估值行业误当成机会。

### ⚠ 04. 嫌价值因子慢,刚买没涨就割肉换股

价值因子是个慢因子,有时灵有时不灵,可能连续几年跑输。沉不住气、刚买没涨就频繁换股,正好把它最需要的耐心给丢了。

### ⚠ 05. 把便宜当成不会跌的保险

很多人以为「都这么便宜了还能跌到哪去」,可低估值股照样能再腰斩。便宜不是地板,越跌越加仓死扛,容易越套越深。

## 实战 SOP · 用价值因子选股 — 几条铁规矩

1. 先想清楚价值因子在赌什么:赌被市场低估的票迟早价值回归,它是挑相对便宜的一篮子,不是预测明天涨跌。
2. 估值尺子要选对:看回本年数用市盈率,看家底折价用市净率,跨负债比公司用 EV 除以息税折旧前利润。
3. 比估值必须同行业内比:银行跟银行比、科技跟科技比,绝不把天然低PB的行业和天然高PB的行业混在一起排序。
4. 永远多问一句「为什么便宜」:是市场错杀(机会),还是家底在缩水、行业在萎缩(价值陷阱),区分清楚再下手。
5. 给价值因子足够的耐心:它是慢因子,可能连续几年跑输,沉得住气熬得过去,才能等到价值回归那一天。

> 把这段打印贴在你电脑边,执行 1000 次它会回报你。

## 总结 · 你应该带走的

2. 价值因子=用市净率给票打分,越便宜(市净率越低)分越高,赌便宜的东西迟早涨回来。
3. 市净率=股价相对公司净资产的倍数:0.7倍是打七折的便宜票,15倍是市场溢价押注成长的贵票。
4. 诚实的反直觉结果:2015-2024这10 年,低PB便宜组+136.8%反而跑输高PB贵组+372.7%,A股成长牛市里便宜不等于会涨。
5. 价值因子有周期、要极大耐心:逐年超额10年只有4年为正,2020年单年跑输188个点,2021年起才陆续翻身。
6. 警惕价值陷阱:金地集团常年市净率约1.1倍看着便宜,10 年却亏44.5%、股价一路阴跌,便宜是有原因的。
7. 便宜组跌得少也涨得少:最大回撤-31.6%远小于贵组-67.1%,价值因子更像防守,在扛跌控回撤上有价值。

## 自测题

**Q1.** 价值因子本质上在做一件什么事?它是在预测股票明天涨还是跌,还是在挑相对便宜的一篮子赌价值回归?请用自己的话说清楚。

**Q2.** 市净率 PB 是怎么算的?如果一只票的市净率是零点8 倍,用大白话解释这意味着你花几块钱买它1 块钱的什么?为什么银行、钢铁这类行业天然就比科技、医药低?

**Q3.** 什么是价值陷阱?为什么有的票一直便宜却一直跌、越买越亏?光看一个低市净率就买进去,可能漏看了什么关键问题?

把答案写下来,3 天后再回看。

## 下一节预告

**Day 069 · 成长因子** (Growth Factors)

这一节我们用市净率捡便宜,也认清了价值陷阱:便宜是有原因的。下一节我们换个完全相反的视角,去追成长,看营收增速、看利润的复合年化增长率,把高成长的票和低成长的票分开回测,看看为高成长付出的溢价到底值不值,以及成长股一旦增速失速会摔得多惨。

## 推荐阅读

- Eugene Fama, Kenneth French《Common Risk Factors in the Returns on Stocks and Bonds》(1993/JFE) — 三因子模型奠基之作,HML 价值因子(高账面市值比减低账面市值比)从这里来,价值投资量化化的源头。
- Benjamin Graham, David Dodd《Security Analysis》(1934/McGraw-Hill) — 价值投资圣经,第一次系统讲清「用低于内在价值的价格买入」,所有价值因子的思想原点。
- Joseph Piotroski《Value Investing: The Use of Historical Financial Statement Information》(2000/JAR) — 提出 F-Score,教你在便宜股里用财务质量筛掉价值陷阱,正好补价值因子的最大短板。
- Aswath Damodaran《The Little Book of Valuation》(2011/Wiley) — 估值大师写给普通人的小书,把市盈率、市净率、EV 倍数讲得通俗透彻,理解每把尺子适用场景必读。
- Tobias Carlisle《Deep Value》(2014/Wiley) — 专门研究极端便宜股,既讲深度价值的超额收益,也直面价值陷阱怎么识别,价值因子实战派的反面教材合集。